# Reto AgroCebada 2026 · Cuaderno final

Todo el proceso en un solo lugar:
1. Carga y revisión de los datos.
2. Variables.
3. Comparación de 43 modelos con los mismos repartos.
4. **Selección automática del ganador** con la regla acordada.
5. Entrenamiento final y las 59 predicciones.
6. Gráficas para el reporte.
7. Comparación contra los rendimientos reales, cuando FIRA (Fideicomisos Instituidos en Relación con la Agricultura) los entregue.

**Regla de selección (fijada antes de correr):**
1. Gana el modelo con menor **RMSE** (raíz del error cuadrático medio) promedio en los repartos 42 a 51.
2. Si varios **empatan** (la diferencia con el mejor es menor que lo que esa diferencia varía de un reparto a otro), gana el de menor **MAE** (error absoluto medio).
3. Si en MAE también empatan, gana el de **menor variación** del RMSE entre repartos.

R² (coeficiente de determinación) se reporta como apoyo.

**Todos los modelos pasan exactamente las mismas pruebas.** Los repartos 52 a 61 sirven solo para confirmar.

**Qué subir:** `todo__basico_terminado.csv` y `ID_area_rendimiento_70_30_Reto_AgroCebada.csv` (archivo original de FIRA, sin abrirlo ni guardarlo en Excel). Opcionales: `candidatas_fase2.csv` y `diccionario_1381_completo.csv`, solo para verificaciones.

**Tiempo medido:** con `COMPARAR = True`, la sección 7 tardó 35.5 minutos en Colab. Con `COMPARAR = False` (solo los 19 modelos del primer grupo de la sección 6), todo el cuaderno tardó 7.5 minutos en una prueba fuera de Colab.

## 1 · Configuración

In [ ]:
COMPARAR = True                         # True: compara los 43 modelos · False: solo los 19 del primer grupo de la sección 6 (rápido)
SEMILLA = 42
SEMILLAS = list(range(42, 52))          # repartos para comparar y seleccionar
SEMILLAS_NUEVAS = list(range(52, 62))   # repartos para confirmar (no se usan para elegir)
N_ALFAS = 40                            # penalizaciones que prueban ElasticNet y Lasso
MEZCLAS = [0.1, 0.5, 0.9]               # mezclas de ElasticNet
PASO_W = 0.05                           # pesos de las parejas: 0, 0.05, ..., 1
N_BOOT = 2000                           # remuestreos de parcelas
TOPO = ["elev_mean", "pend_mean"]       # elevación y pendiente
MESES_LSTM = list(range(4, 11))         # abril a octubre (acordado: la LSTM no usa meses fuera del ciclo)
MESES_CICLO = list(range(4, 11))        # clima mensual: abril a octubre de 2025
COL_LLUVIA = ["prec_2025_%02d_mean" % m for m in MESES_CICLO]
COL_TEMP = ["%s_2025_%02d_mean" % (c, m) for c in ("tmin", "tmax") for m in MESES_CICLO]
COL_CLIMA = COL_LLUVIA + COL_TEMP
COL_TMAX = ["tmax_2025_%02d_mean" % m for m in MESES_CICLO]
COL_TMIN = ["tmin_2025_%02d_mean" % m for m in MESES_CICLO]
COL_TEMP_ED = ["%s_2025_%02d_mean" % (c, m) for c in ("tmin", "tmax") for m in range(1, 13)]     # enero–diciembre
COL_CLIMA_ED = ["%s_2025_%02d_mean" % (c, m) for c in ("prec", "tmin", "tmax") for m in range(1, 13)]
COL_TEMP_EO = ["%s_2025_%02d_mean" % (c, m) for c in ("tmin", "tmax") for m in range(1, 11)]     # enero–octubre
MEZCLAS_AMPLIA = [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
MODELOS = ["XGBoost", "Random Forest", "ElasticNet", "Lasso", "LSTM"]

NOMBRE_EQUIPO = "visionterradata"       # archivo de entrega: visionterradata_prediccion.csv (correo de FIRA del 14 de octubre)
ARCHIVO_FIRA_ORIGINAL = "ID_area_rendimiento_70_30_Reto_AgroCebada.csv"   # archivo original de FIRA, con las 197 parcelas
HUELLA_FIRA_ORIGINAL = "c4ea664d9c6161fcfd8f297d6f53db9a"   # huella md5 (algoritmo de resumen de mensajes 5) del archivo original revisado
COLUMNAS_FIRA = ["ID_POLIGONO", "AREA_HA", "RENDIMIENTO_T_HA", "CONJUNTO"]   # nombres y orden de las columnas del archivo de FIRA
COLUMNA_ID = "ID_parcela"               # encabezados de la tabla del anexo (Tabla 3)
COLUMNA_REND = "rendimiento_t_ha"
DECIMALES = 4                           # cuatro decimales (FIRA, segunda sesión de preguntas, respuesta 7)
# Huella md5 de las 59 predicciones ya validadas, redondeadas a dos decimales (visionterradata_prediccion.csv
# del paquete del 7 de octubre). La sección 12 la compara para comprobar que el modelo no cambió.
HUELLA_VALIDADA_2D = "d94bf3820745bcbd57bba94167e3994d"

import os, re, sys, io, json, shutil, hashlib, itertools, warnings, subprocess, time, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import enet_path
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.exceptions import ConvergenceWarning
for paquete in ("xgboost", "jax"):
    try:
        __import__(paquete)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)
from xgboost import XGBRegressor
import jax, jax.numpy as jnp
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # comprobado: no cambia resultados
pd.set_option("display.width", 220)
PAREJAS = list(itertools.combinations(MODELOS, 2))
assert len(PAREJAS) == 10 and len({frozenset(p) for p in PAREJAS}) == 10
print("Listo · COMPARAR =", COMPARAR)

## 2 · Carga y revisión

Cuando aparezca **"Elegir archivos"**, sube los dos archivos juntos:
1. `todo__basico_terminado.csv`, la tabla maestra.
2. `ID_area_rendimiento_70_30_Reto_AgroCebada.csv`, el archivo original de FIRA (Fideicomisos Instituidos en Relación con la Agricultura). **No lo abras ni lo guardes en Excel antes de subirlo:** de él se copian tal cual el identificador, el área y el conjunto del CSV (valores separados por comas) de entrega, y el cuaderno comprueba que sea idéntico al original y que coincida con la tabla maestra.

Si llegan con otro nombre (espacios, "+", "(1)"), el cuaderno los reconoce solo.

In [ ]:
NECESARIOS = ["todo__basico_terminado.csv", ARCHIVO_FIRA_ORIGINAL]
OPCIONALES = ["candidatas_fase2.csv", "diccionario_1381_completo.csv"]

def acomodar_nombres():
    clave = lambda n: re.sub(r"[^a-z0-9]", "", re.sub(r"\s*\(\d+\)$", "", os.path.splitext(n)[0].lower()))
    for f in NECESARIOS + OPCIONALES:
        if os.path.exists(f):
            continue
        parecidos = sorted([n for n in os.listdir(".") if n.lower().endswith(".csv") and clave(n) == clave(f)],
                           key=os.path.getmtime)
        if parecidos:
            shutil.copy(parecidos[-1], f)
            print("Se usó '%s' como %s" % (parecidos[-1], f))

acomodar_nombres()
if not all(os.path.exists(f) for f in NECESARIOS):
    try:
        from google.colab import files
        print("Sube todo__basico_terminado.csv y %s (opcionales: candidatas_fase2.csv y diccionario_1381_completo.csv)" % ARCHIVO_FIRA_ORIGINAL)
        files.upload()
    except ImportError:
        pass
    acomodar_nombres()
faltan = [f for f in NECESARIOS if not os.path.exists(f)]
if faltan:
    raise Exception("Faltan archivos (deben llamarse exactamente así): " + ", ".join(faltan))

d = pd.read_csv("todo__basico_terminado.csv", low_memory=False)
ent = d.RENDIMIENTO_T_HA.notna().values
fallos = []
if d.shape != (197, 1381): fallos.append("tabla maestra: %s" % (d.shape,))
if ent.sum() != 138: fallos.append("no son 138 parcelas con rendimiento")
if d.ID_POLIGON.duplicated().any(): fallos.append("identificadores repetidos")
if (d.loc[~ent, "CONJUNTO"] != "PREDICCION").any(): fallos.append("parcelas sin rendimiento fuera de PREDICCION")
if fallos:
    raise Exception("VERIFICACIÓN FALLIDA:\n- " + "\n- ".join(fallos))
y, g = d.RENDIMIENTO_T_HA.values[ent], d.Municipio.values[ent]
print("Revisión: OK · %d parcelas · %d con rendimiento · %d a predecir · %d municipios"
      % (len(d), ent.sum(), (~ent).sum(), d.Municipio.nunique()))
print("Rendimiento de entrenamiento: %.2f a %.2f t/ha, promedio %.3f" % (y.min(), y.max(), y.mean()))
print("Opcionales presentes:", [f for f in OPCIONALES if os.path.exists(f)] or "ninguno")

# Archivo original de FIRA: de aquí se copian tal cual el identificador, el área y el conjunto del CSV de entrega
with open(ARCHIVO_FIRA_ORIGINAL, "rb") as fh:
    BYTES_FIRA = fh.read()
if hashlib.md5(BYTES_FIRA).hexdigest() != HUELLA_FIRA_ORIGINAL:
    raise Exception("%s no es idéntico al archivo original de FIRA que revisamos (huella md5 distinta). Vuelve a "
                    "descargarlo y súbelo sin abrirlo ni guardarlo en Excel. Si FIRA mandó una versión nueva, "
                    "hay que revisarla antes de continuar." % ARCHIVO_FIRA_ORIGINAL)
fira = pd.read_csv(io.BytesIO(BYTES_FIRA), dtype=str, keep_default_na=False, encoding="utf-8-sig")
vacias = [c for c in fira.columns if c.startswith("Unnamed")]    # la coma al final de cada renglón crea una columna vacía
fallos = []
if (fira[vacias] != "").any().any(): fallos.append("la columna sin nombre del final trae datos")
fira = fira.drop(columns=vacias)
if list(fira.columns) != COLUMNAS_FIRA:
    fallos.append("columnas del archivo de FIRA: %s" % list(fira.columns))
elif len(fira) != 197 or fira.ID_POLIGONO.duplicated().any() or set(fira.ID_POLIGONO) != set(d.ID_POLIGON):
    fallos.append("los identificadores del archivo de FIRA no son los 197 de la tabla maestra")
else:
    F_, M_ = fira.set_index("ID_POLIGONO"), d.set_index("ID_POLIGON").loc[fira.ID_POLIGONO]   # mismo orden
    pred_ = (F_.CONJUNTO == "PREDICCION").values
    if set(F_.index[pred_]) != set(d.ID_POLIGON[~ent]): fallos.append("las parcelas de PREDICCION no son las 59 sin rendimiento")
    if list(F_.CONJUNTO) != list(M_.CONJUNTO): fallos.append("el conjunto no coincide con la tabla maestra")
    igual = lambda a, b: np.allclose(pd.to_numeric(a, errors="coerce"), b, rtol=0, atol=1e-9)    # un vacío cuenta como diferencia
    if not igual(F_.AREA_HA, M_["área_ha"].values): fallos.append("el área no coincide con la tabla maestra")
    if (F_.RENDIMIENTO_T_HA[pred_] != "").any(): fallos.append("hay rendimientos en parcelas de PREDICCION")
    if not igual(F_.RENDIMIENTO_T_HA[~pred_], M_.RENDIMIENTO_T_HA.values[~pred_]):
        fallos.append("el rendimiento de entrenamiento no coincide con la tabla maestra")
if fallos:
    raise Exception("VERIFICACIÓN FALLIDA (archivo de FIRA):\n- " + "\n- ".join(fallos))
print("Archivo de FIRA: OK · idéntico al original · %d parcelas · %d de PREDICCION · identificador, conjunto, área y "
      "rendimiento coinciden con la tabla maestra" % (len(fira), pred_.sum()))

## 3 · Preparación y selección de variables

Se rehacen las 22 variables desde la tabla maestra, con las mismas funciones de las Fases 1 y 2:

1. Clima y fenología de cada índice en dos ventanas: 81 variables.
2. Corte 1: se queda la ventana oficial, abril–octubre.
3. Corte 2: se quita el LAI (índice de área foliar).
4. Corte 3: de cada grupo de variables que se parecen más de 95 %, se deja una.

**Ningún corte mira el rendimiento.**

In [ ]:
VENTANAS = {"abroct": {"ciclo": [4, 5, 6, 7, 8, 9, 10], "base": [1, 2, 3]},
            "junnov": {"ciclo": [6, 7, 8, 9, 10, 11],   "base": [1, 2, 3, 4, 5]}}
INDICES_PRO, INDICES_BAS = ["ndvi", "evi", "lai", "msavi"], ["ndvi", "evi", "lai"]

def clima(d, etiqueta, meses):                                   # Fase 1, sin cambios
    out = pd.DataFrame(index=d.index)
    out["prec_acum_" + etiqueta] = d[["prec_2025_%02d_mean" % m for m in meses]].sum(axis=1)
    tmin = d[["tmin_2025_%02d_mean" % m for m in meses]].mean(axis=1)
    tmax = d[["tmax_2025_%02d_mean" % m for m in meses]].mean(axis=1)
    out["tmin_media_" + etiqueta] = tmin
    out["tmax_media_" + etiqueta] = tmax
    out["amplitud_termica_" + etiqueta] = tmax - tmin
    return out

def fenologia(d, prefijo, indice, etiqueta, ciclo, base):        # Fase 1, sin cambios
    col = lambda m: "%s%s_2025_%02d_mean" % (prefijo, indice, m)
    C, B = d[[col(m) for m in ciclo]], d[[col(m) for m in base]]
    n = "%s%s_%s" % (prefijo, indice, etiqueta)
    pico, linea = C.max(axis=1), B.mean(axis=1)
    return pd.DataFrame({n + "_pico": pico,
                         n + "_mes_pico": C.idxmax(axis=1).str.extract(r"_2025_(\d\d)_")[0].astype(float),
                         n + "_integral": C.mean(axis=1) * len(ciclo),
                         n + "_base": linea, n + "_amplitud": pico - linea}, index=d.index)

partes = [d[["elev_mean", "pend_mean"]]]
for etq, v in VENTANAS.items():
    partes.append(clima(d, etq, v["ciclo"]))
partes.append(pd.DataFrame({"prec_acum_presiembra": d[["prec_2025_%02d_mean" % m for m in (3, 4, 5)]].sum(axis=1)}, index=d.index))
for etq, v in VENTANAS.items():
    for ix in INDICES_PRO: partes.append(fenologia(d, "", ix, etq, v["ciclo"], v["base"]))
    for ix in INDICES_BAS: partes.append(fenologia(d, "bas_", ix, etq, v["ciclo"], v["base"]))
CAND = pd.concat(partes, axis=1)
print("Candidatas:", CAND.shape[1], "variables")

PRIO_INDICE = {"ndvi": 1, "evi": 2, "msavi": 3, "lai": 4}       # Fase 2, sin cambios
PRIO_DERIVADA = {"pico": 1, "integral": 2, "amplitud": 3, "base": 4, "mes_pico": 5}
def prioridad(v):
    if v.startswith(("elev_", "pend_")): return (0, 0, 0, v)
    if v.startswith(("prec_", "tmin_", "tmax_")): return (0, 1, 0, v)
    if v.startswith("amplitud_termica"): return (0, 2, 0, v)
    p = v[4:] if v.startswith("bas_") else v
    der = "mes_pico" if p.endswith("mes_pico") else p.split("_")[-1]
    return (1, PRIO_INDICE.get(p.split("_")[0], 9), PRIO_DERIVADA.get(der, 9), v)
def corte_3(X, umbral=0.95):
    V = list(X.columns); C = X.corr().abs(); padre = {v: v for v in V}
    def raiz(a):
        while padre[a] != a:
            padre[a] = padre[padre[a]]; a = padre[a]
        return a
    for i, a in enumerate(V):
        for b in V[i + 1:]:
            if C.loc[a, b] > umbral: padre[raiz(a)] = raiz(b)
    grupos = {}
    for v in V: grupos.setdefault(raiz(v), []).append(v)
    return sorted([sorted(m, key=prioridad)[0] for m in grupos.values()], key=prioridad)

c1_ = CAND.drop(columns=[c for c in CAND.columns if "_junnov" in c])
c2_ = c1_.drop(columns=[c for c in c1_.columns if c.startswith(("lai_", "bas_lai_"))])
VARS = corte_3(c2_)
print("Corte 1: %d · Corte 2: %d · Corte 3: %d variables" % (c1_.shape[1], c2_.shape[1], len(VARS)))
if os.path.exists("candidatas_fase2.csv"):
    ref = pd.read_csv("candidatas_fase2.csv", encoding="utf-8-sig")
    otras = [c for c in ref.columns if c not in ("ID_POLIGON", "Municipio", "Estado", "CONJUNTO", "área_ha", "RENDIMIENTO_T_HA")]
    dif = float((ref[otras].values - CAND[otras].values).__abs__().max())
    print("Verificación contra candidatas_fase2.csv: mismas 22 = %s · diferencia máxima %.1e" % (set(otras) == set(VARS), dif))
TOPO = ["elev_mean", "pend_mean"]
X_all = CAND[VARS].values.astype(float)

## 4 · Correlaciones

Solo descriptivo: **no se usa para elegir variables**.

- **Mapa de calor:** las 22 variables y el rendimiento, con las 138 parcelas.
- **Tabla:** la correlación con el rendimiento usando todas las parcelas, contra la que queda dentro del municipio (promedio en los municipios con más de 3 parcelas).

In [ ]:
T = CAND.loc[ent, VARS].assign(RENDIMIENTO_T_HA=y)
C = T[["RENDIMIENTO_T_HA"] + VARS].corr()
fig, ax = plt.subplots(figsize=(11, 11))
im = ax.imshow(C.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(C))); ax.set_xticklabels(C.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(C))); ax.set_yticklabels(C.columns, fontsize=8)
for i in range(len(C)):
    for j in range(len(C)):
        ax.text(j, i, "%.2f" % C.values[i, j], ha="center", va="center", fontsize=5.5,
                color="white" if abs(C.values[i, j]) > 0.7 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="correlación")
ax.set_title("Correlaciones · 22 variables y rendimiento · 138 parcelas")
plt.tight_layout(); fig.savefig("mapa_calor_22.png", dpi=150); plt.show()

grandes = [m for m in pd.Series(g).value_counts().index if (g == m).sum() > 3]
filas = []
for v in VARS:
    x = T[v].values
    dentro = [np.corrcoef(x[g == m], y[g == m])[0, 1] for m in grandes if np.std(x[g == m]) > 0]
    filas.append({"variable": v, "con todas las parcelas": np.corrcoef(x, y)[0, 1], "dentro del municipio": np.mean(dentro)})
CORR = pd.DataFrame(filas).round(3)
CORR = CORR.reindex(CORR["con todas las parcelas"].abs().sort_values(ascending=False).index).reset_index(drop=True)
print("Municipios usados para 'dentro del municipio':", grandes)
CORR

## 5 · Secuencias mensuales para la LSTM (memoria de corto y largo plazo)

Cada parcela es una serie de **7 meses (abril a octubre de 2025) con 7 datos por mes**: lluvia, temperatura mínima y máxima, y NDVI y EVI (índices de vegetación) de Planet y de Sentinel-2. Ya no usa los 12 meses.

In [ ]:
CANALES = ["prec", "tmin", "tmax", "ndvi", "evi", "bas_ndvi", "bas_evi"]
S_all = np.stack([d[["%s_2025_%02d_mean" % (c, m) for m in MESES_LSTM]].values for c in CANALES], axis=2).astype(float)
assert not np.isnan(S_all).any(), "hay vacíos en las secuencias"
print("Secuencias:", S_all.shape, "(parcelas × meses abril–octubre × datos por mes) · sin vacíos")
if os.path.exists("diccionario_1381_completo.csv"):
    dic = pd.read_csv("diccionario_1381_completo.csv", encoding="utf-8-sig").set_index("nombre")
    for c in CANALES:
        n = "%s_2025_07_mean" % c
        print("   %-9s ->  %s" % (c, dic.loc[n, "significado"] if n in dic.index else "(no está en el diccionario)"))

## 6 · Los 43 modelos

| Grupo | Modelos |
|---|---|
| Con promedio municipal | Promedio municipal · + ½ topografía · + topografía completa · + ½ clima mensual · + clima mensual completo · + w·clima mensual · + ½ temperatura mensual (máx. y mín.) · + temperatura completa · + w·temperatura · + ½ temperatura máxima · + ½ temperatura mínima · + ½ lluvia mensual · + ½ temperatura enero–diciembre · + ½ clima enero–diciembre · + ½ temperatura enero–octubre · + ½ temperatura (más mezclas) · + ½ temperatura + ½ topografía (separadas) · + ½ (temperatura y topografía juntas) · Solo topografía (sin municipio) |
| Individuales (22 variables o series) | XGBoost · Random Forest · ElasticNet · Lasso · LSTM |
| Con clima mensual | Random Forest · XGBoost · Vecinos parecidos |
| Parejas | Las 10 de los individuales, más (Municipal + ½ clima) con cada individual y con (Municipal + ½ topografía): w elegido dentro del entrenamiento |

- Los modelos con municipio aprenden la diferencia contra el promedio de su municipio, calculado solo con el entrenamiento.
- "Topografía" usa ElasticNet con elevación y pendiente; "clima mensual" usa lluvia y temperatura mínima y máxima de cada mes, de abril a octubre; "temperatura mensual" usa solo las temperaturas. Salvo que se diga otra cosa, todo es de abril a octubre.
- "½" significa promedio municipal + 0.5 × el ajuste de ElasticNet; "w" es un peso entre 0 y 1 elegido con validación interna.
- Los modelos individuales usan las 22 variables; la LSTM usa las series de abril a octubre.

In [ ]:
def internos(n):
    return list(KFold(5, shuffle=True, random_state=SEMILLA).split(np.arange(n)))

def centros(Z, y_, g_):
    gs = sorted(set(g_))
    return ({m: Z[g_ == m].mean(axis=0) for m in gs}, {m: y_[g_ == m].mean() for m in gs}, Z.mean(axis=0), y_.mean())
def quitar(Z, g_, mz, gz):  return Z - np.stack([mz.get(m, gz) for m in g_])
def base_y(g_, my, gy):     return np.array([my.get(m, gy) for m in g_])

# --- ElasticNet y Lasso (lógica de la Fase 3; además devuelve las predicciones internas) ---
def _camino(X, y_, g_, alfas, l1):
    mz, my, gz, gy = centros(X, y_, g_)
    Xw, yw = quitar(X, g_, mz, gz), y_ - base_y(g_, my, gy)
    s = Xw.std(axis=0); s[s == 0] = 1.0
    _, coefs, _ = enet_path(Xw / s, yw, l1_ratio=l1, alphas=alfas, max_iter=20000, tol=1e-4)
    return dict(B=coefs / s[:, None], mz=mz, my=my, gz=gz, gy=gy)
def _pred_lin(aj, X, g_):
    return base_y(g_, aj["my"], aj["gy"])[:, None] + quitar(X, g_, aj["mz"], aj["gz"]) @ aj["B"]
def _rejilla(X, y_, g_, l1):
    mz, my, gz, gy = centros(X, y_, g_)
    Xw, yw = quitar(X, g_, mz, gz), y_ - base_y(g_, my, gy)
    s = Xw.std(axis=0); s[s == 0] = 1.0
    amax = np.abs((Xw / s).T @ yw).max() / (len(y_) * l1)
    return np.logspace(np.log10(amax), np.log10(amax * 1e-3), N_ALFAS)
def lineal(X, y_, g_, Xte, gte, l1s):
    mejor = None
    for l1 in l1s:
        alfas = _rejilla(X, y_, g_, l1); oof = np.zeros((len(y_), len(alfas)))
        for tr, te in internos(len(y_)):
            oof[te] = _pred_lin(_camino(X[tr], y_[tr], g_[tr], alfas, l1), X[te], g_[te])
        err = ((oof - y_[:, None]) ** 2).sum(axis=0); k = int(err.argmin())
        if mejor is None or err[k] < mejor[0]:
            mejor = (err[k], l1, alfas[k], oof[:, k].copy())
    _, l1, alfa, oof = mejor
    return _pred_lin(_camino(X, y_, g_, np.array([alfa]), l1), Xte, gte)[:, 0], oof, "l1=%.1f" % l1

# --- XGBoost y Random Forest: aprenden la diferencia contra su municipio ---
def _arbol(hacer, X, y_, g_, Xte, gte):
    mz, my, gz, gy = centros(X, y_, g_)
    m = hacer(); m.fit(quitar(X, g_, mz, gz), y_ - base_y(g_, my, gy))
    return base_y(gte, my, gy) + m.predict(quitar(Xte, gte, mz, gz))
def arboles(fabricas, X, y_, g_, Xte, gte):
    mejor = None
    for nombre, hacer in fabricas.items():
        oof = np.zeros(len(y_))
        for tr, te in internos(len(y_)):
            oof[te] = _arbol(hacer, X[tr], y_[tr], g_[tr], X[te], g_[te])
        err = ((oof - y_) ** 2).sum()
        if mejor is None or err < mejor[0]:
            mejor = (err, nombre, oof)
    _, nombre, oof = mejor
    return _arbol(fabricas[nombre], X, y_, g_, Xte, gte), oof, nombre
XGB = {"prof%d_hoja%d" % (p, h): (lambda p=p, h=h: XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=p,
        min_child_weight=h, subsample=0.8, colsample_bytree=0.8, random_state=SEMILLA, n_jobs=1))
       for p in (2, 3) for h in (5, 10)}
RF = {"hoja%d" % h: (lambda h=h: RandomForestRegressor(n_estimators=200, min_samples_leaf=h, max_features=0.5,
        random_state=SEMILLA, n_jobs=-1)) for h in (5, 10)}

# --- LSTM en JAX: 8 unidades, 200 épocas, Adam; aprende la diferencia contra su municipio ---
NMAX, H_LSTM, EPOCAS, LR = 138, 8, 200, 0.01
LAMS = [0.01, 0.1, 1.0]
def _iniciar(c, h, clave):
    k1, k2, k3 = jax.random.split(clave, 3)
    return dict(Wx=jax.random.normal(k1, (c, 4 * h)) / np.sqrt(c), Wh=jax.random.normal(k2, (h, 4 * h)) / np.sqrt(h),
                b=jnp.zeros(4 * h).at[h:2 * h].set(1.0), Wo=jax.random.normal(k3, (h,)) * 0.1, bo=jnp.zeros(()))
def _adelante(p, S):
    h0 = jnp.zeros((S.shape[0], p["Wh"].shape[0]))
    def paso(estado, x):
        h, c = estado
        i, f, u, o = jnp.split(x @ p["Wx"] + h @ p["Wh"] + p["b"], 4, axis=1)
        c = jax.nn.sigmoid(f) * c + jax.nn.sigmoid(i) * jnp.tanh(u)
        return (jax.nn.sigmoid(o) * jnp.tanh(c), c), None
    (h, _), _ = jax.lax.scan(paso, (h0, h0), jnp.swapaxes(S, 0, 1))
    return h @ p["Wo"] + p["bo"]
def _perdida(p, S, r, w, lam):
    reg = sum(jnp.sum(p[k] ** 2) for k in ("Wx", "Wh", "Wo"))
    return jnp.sum(w * (_adelante(p, S) - r) ** 2) / jnp.sum(w) + lam * reg
@jax.jit
def _entrenar(p, S, r, w, lam):
    grad = jax.grad(_perdida)
    def paso(estado, t):
        p, m, v = estado
        dg = grad(p, S, r, w, lam)
        m = jax.tree_util.tree_map(lambda a, b: 0.9 * a + 0.1 * b, m, dg)
        v = jax.tree_util.tree_map(lambda a, b: 0.999 * a + 0.001 * b * b, v, dg)
        p = jax.tree_util.tree_map(lambda a, mm, vv: a - LR * (mm / (1 - 0.9 ** t)) / (jnp.sqrt(vv / (1 - 0.999 ** t)) + 1e-8), p, m, v)
        return (p, m, v), None
    ceros = jax.tree_util.tree_map(jnp.zeros_like, p)
    (p, _, _), _ = jax.lax.scan(paso, (p, ceros, ceros), jnp.arange(1, EPOCAS + 1, dtype=jnp.float32))
    return p
_pred_lstm = jax.jit(_adelante)
def _lstm_uno(S, y_, g_, Ste, gte, lam):
    mz, my, gz, gy = centros(S, y_, g_)
    Sw, r = quitar(S, g_, mz, gz), y_ - base_y(g_, my, gy)
    esc = Sw.reshape(-1, S.shape[2]).std(axis=0); esc[esc == 0] = 1.0
    er = r.std() if r.std() > 0 else 1.0
    n = len(y_); Sp = np.zeros((NMAX,) + S.shape[1:]); Sp[:n] = Sw / esc     # tamaño fijo: se compila una sola vez
    rp = np.zeros(NMAX); rp[:n] = r / er; wp = np.zeros(NMAX); wp[:n] = 1.0
    p = _entrenar(_iniciar(S.shape[2], H_LSTM, jax.random.PRNGKey(SEMILLA)), jnp.asarray(Sp, jnp.float32),
                  jnp.asarray(rp, jnp.float32), jnp.asarray(wp, jnp.float32), jnp.float32(lam))
    Tw = quitar(Ste, gte, mz, gz) / esc; Tp = np.zeros((NMAX,) + S.shape[1:]); Tp[:len(Tw)] = Tw
    return base_y(gte, my, gy) + np.asarray(_pred_lstm(p, jnp.asarray(Tp, jnp.float32)))[:len(Tw)] * er
def lstm(S, y_, g_, Ste, gte):
    mejor = None
    for lam in LAMS:
        oof = np.zeros(len(y_))
        for tr, te in internos(len(y_)):
            oof[te] = _lstm_uno(S[tr], y_[tr], g_[tr], S[te], g_[te], lam)
        err = ((oof - y_) ** 2).sum()
        if mejor is None or err < mejor[0]:
            mejor = (err, lam, oof)
    _, lam, oof = mejor
    return _lstm_uno(S, y_, g_, Ste, gte, lam), oof, "lam=%g" % lam


PESOS = np.round(np.arange(0, 1 + 1e-9, PASO_W), 2)
GANADOR_VALIDADO = "Promedio municipal + ½ temperatura máxima mensual"   # solo para comprobar cifras; NO interviene en la selección
from sklearn.neighbors import KNeighborsRegressor

# Matrices para las 197 parcelas; cada modelo usa solo las filas de entrenamiento que se le pasan
IDX_ENT, IDX_PRED = np.where(ent)[0], np.where(~ent)[0]
Y_ALL, G_ALL = d.RENDIMIENTO_T_HA.values.astype(float), d.Municipio.values
XA, XT_, S_ = X_all, d[TOPO].values.astype(float), S_all
XC_, XTT_, XP_ = d[COL_CLIMA].values.astype(float), d[COL_TEMP].values.astype(float), d[COL_LLUVIA].values.astype(float)
XTX_, XTN_ = d[COL_TMAX].values.astype(float), d[COL_TMIN].values.astype(float)
XTED_, XCED_, XTEO_ = d[COL_TEMP_ED].values.astype(float), d[COL_CLIMA_ED].values.astype(float), d[COL_TEMP_EO].values.astype(float)
XTTOPO_ = np.column_stack([XTT_, XT_])
y_ent, g_ent = Y_ALL[IDX_ENT], G_ALL[IDX_ENT]

def municipal_oof(ytr, gtr):
    o = np.zeros(len(ytr))
    for a, b in internos(len(ytr)):
        _, my, _, gy = centros(np.zeros((len(a), 1)), ytr[a], gtr[a]); o[b] = base_y(gtr[b], my, gy)
    return o
def vecinos(X, ytr, gtr, Xte, gte):
    """Promedio municipal + diferencia promedio de los k vecinos más parecidos (k elegido por validación interna)."""
    def uno(k, X, ytr, gtr, Xte, gte):
        mz, my, gz, gy = centros(X, ytr, gtr)
        Xw = quitar(X, gtr, mz, gz); sc = Xw.std(axis=0); sc[sc == 0] = 1
        m = KNeighborsRegressor(n_neighbors=min(k, len(ytr))).fit(Xw / sc, ytr - base_y(gtr, my, gy))
        return base_y(gte, my, gy) + m.predict(quitar(Xte, gte, mz, gz) / sc)
    mejor = None
    for k in (5, 10, 20, 40):
        oof = np.zeros(len(ytr))
        for a, b in internos(len(ytr)):
            oof[b] = uno(k, X[a], ytr[a], gtr[a], X[b], gtr[b])
        e = ((oof - ytr) ** 2).sum()
        if mejor is None or e < mejor[0]: mejor = (e, k, oof)
    return uno(mejor[1], X, ytr, gtr, Xte, gte), mejor[2], "k=%d" % mejor[1]
def peso(oa, ob, ytr):
    return PESOS[int(np.argmin([((w * oa + (1 - w) * ob - ytr) ** 2).sum() for w in PESOS]))]

MUN, CLI = "Promedio municipal", "Promedio municipal + ½ clima mensual"
def ajustar_todo(rtr, rte):
    """Un pliegue: todos los modelos se ajustan solo con las filas 'rtr' y predicen las filas 'rte'."""
    ytr, gtr, gte = Y_ALL[rtr], G_ALL[rtr], G_ALL[rte]
    R, O, E = {}, {}, {}
    _, my, _, gy = centros(np.zeros((len(rtr), 1)), ytr, gtr)
    R[MUN], O[MUN] = base_y(gte, my, gy), municipal_oof(ytr, gtr)
    lin = lambda X, l1s=MEZCLAS: lineal(X[rtr], ytr, gtr, X[rte], gte, l1s)
    topo, clima, temp, lluvia = lin(XT_), lin(XC_), lin(XTT_), lin(XP_)
    uno_tr, uno_te = np.array(["x"] * len(rtr)), np.array(["x"] * len(rte))
    R["Solo topografía (sin municipio)"] = lineal(XT_[rtr], ytr, uno_tr, XT_[rte], uno_te, MEZCLAS)[0]
    R["Promedio municipal + topografía completa"] = topo[0]
    R["Promedio municipal + clima mensual completo"] = clima[0]
    tmax, tmin, temp_ed, clima_ed, temp_eo = lin(XTX_), lin(XTN_), lin(XTED_), lin(XCED_), lin(XTEO_)
    temp_amplia, temp_topo = lin(XTT_, MEZCLAS_AMPLIA), lin(XTTOPO_)
    for nombre, m in (("Promedio municipal + ½ topografía", topo), (CLI, clima),
                      ("Promedio municipal + ½ temperatura mensual (máx. y mín.)", temp), ("Promedio municipal + ½ lluvia mensual", lluvia),
                      ("Promedio municipal + ½ temperatura máxima mensual", tmax), ("Promedio municipal + ½ temperatura mínima mensual", tmin),
                      ("Promedio municipal + ½ temperatura enero–diciembre", temp_ed), ("Promedio municipal + ½ clima enero–diciembre", clima_ed),
                      ("Promedio municipal + ½ temperatura enero–octubre", temp_eo),
                      ("Promedio municipal + ½ temperatura (más mezclas de ElasticNet)", temp_amplia),
                      ("Promedio municipal + ½ (temperatura y topografía juntas)", temp_topo)):
        R[nombre], O[nombre] = 0.5 * (R[MUN] + m[0]), 0.5 * (O[MUN] + m[1])
    R["Promedio municipal + temperatura mensual completa"] = temp[0]
    R["Promedio municipal + ½ temperatura + ½ topografía (separadas)"] = R[MUN] + 0.5 * (temp[0] - R[MUN]) + 0.5 * (topo[0] - R[MUN])
    w = peso(clima[1], O[MUN], ytr)
    R["Promedio municipal + w·clima mensual (w interno)"] = w * clima[0] + (1 - w) * R[MUN]; E["w clima"] = "w=%.2f" % w
    w = peso(temp[1], O[MUN], ytr)
    R["Promedio municipal + w·temperatura mensual (w interno)"] = w * temp[0] + (1 - w) * R[MUN]; E["w temp"] = "w=%.2f" % w
    if not COMPARAR:
        return R, E
    base = {"XGBoost": arboles(XGB, XA[rtr], ytr, gtr, XA[rte], gte), "Random Forest": arboles(RF, XA[rtr], ytr, gtr, XA[rte], gte),
            "ElasticNet": lin(XA), "Lasso": lin(XA, [1.0]), "LSTM": lstm(S_[rtr], ytr, gtr, S_[rte], gte)}
    for m in MODELOS:
        R[m], O[m], E[m] = base[m]
    R["Random Forest con clima mensual"] = arboles(RF, XC_[rtr], ytr, gtr, XC_[rte], gte)[0]
    R["XGBoost con clima mensual"] = arboles(XGB, XC_[rtr], ytr, gtr, XC_[rte], gte)[0]
    R["Vecinos parecidos en clima mensual"] = vecinos(XC_[rtr], ytr, gtr, XC_[rte], gte)[0]
    parejas = PAREJAS + [(CLI, m) for m in MODELOS] + [(CLI, "Promedio municipal + ½ topografía")]
    for a, b in parejas:
        w = peso(O[a], O[b], ytr)
        nombre = "%s + %s" % (a if a != CLI else "(Municipal + ½ clima)", b if b != "Promedio municipal + ½ topografía" else "(Municipal + ½ topografía)")
        R[nombre], E[nombre] = w * R[a] + (1 - w) * R[b], "w=%.2f" % w
    return R, E

print("Modelos listos ·", 43 if COMPARAR else 19, "modelos a evaluar")

## 7 · Validación: mismos repartos para todos

👉 **Aquí se hace la validación.** En cada reparto, las 138 parcelas se dividen en 5 grupos: se entrena con 4 y se predice el quinto, hasta cubrirlas todas. Son 20 repartos: 42 a 51 para comparar y 52 a 61 para confirmar.

In [ ]:
PRED, ELEG = {}, {}
inicio = time.time()
for s in SEMILLAS + SEMILLAS_NUEVAS:
    P, E = {}, []
    for tr, te in KFold(5, shuffle=True, random_state=s).split(y_ent):
        pr, el = ajustar_todo(IDX_ENT[tr], IDX_ENT[te])
        for k, v in pr.items():
            P.setdefault(k, np.zeros(len(y_ent)))[te] = v
        E.append(el)
    PRED[s], ELEG[s] = P, E
    print("Reparto %d listo · %.1f min" % (s, (time.time() - inicio) / 60))
NOMBRES = list(PRED[SEMILLAS[0]].keys())
print("Modelos evaluados:", len(NOMBRES))

## 8 · Verificaciones de precisión

1. **Sin fuga:** si se alteran los rendimientos de la parte de prueba, las predicciones de esa parte no cambian. Igual que en la prueba 2, se aceptan diferencias menores a 0.000000001 t/ha: son redondeo de la computadora (Random Forest suma sus árboles en paralelo y el orden de la suma varía).
2. **Repetible:** el mismo pliegue, corrido dos veces, da exactamente lo mismo.
3. **Coincide con lo ya validado:** las referencias, ElasticNet y Lasso dan las cifras anteriores.

In [ ]:
tr0, te0 = next(KFold(5, shuffle=True, random_state=SEMILLA).split(y_ent))
p_a, _ = ajustar_todo(IDX_ENT[tr0], IDX_ENT[te0])
Y_GUARDADO = Y_ALL.copy()
Y_ALL[IDX_ENT[te0]] = np.random.default_rng(0).uniform(0, 10, len(te0))      # rendimientos de prueba alterados
p_b, _ = ajustar_todo(IDX_ENT[tr0], IDX_ENT[te0])
Y_ALL[:] = Y_GUARDADO
fuga = max(np.abs(p_a[k] - p_b[k]).max() for k in p_a)
print("1. Sin fuga: cambio máximo = %.1e  ->  %s" % (fuga, "OK" if fuga < 1e-9 else "REVISAR"))
rep = max(np.abs(p_a[k] - PRED[SEMILLA][k][te0]).max() for k in p_a)
print("2. Repetible: diferencia máxima = %.1e  ->  %s" % (rep, "OK" if rep < 1e-9 else "REVISAR"))
VALIDADO = {"Promedio municipal": 0.51260, "Promedio municipal + ½ topografía": 0.50831, "Promedio municipal + topografía completa": 0.50857,
            "Solo topografía (sin municipio)": 0.81135, "Promedio municipal + ½ clima mensual": 0.50393,
            "Promedio municipal + ½ temperatura mensual (máx. y mín.)": 0.49785, GANADOR_VALIDADO: 0.49479,
            "ElasticNet": 0.51341, "Lasso": 0.51447}
for k, v in VALIDADO.items():
    if k in PRED[SEMILLA]:
        r = np.mean([np.sqrt(mean_squared_error(y_ent, PRED[s][k])) for s in SEMILLAS])
        print("3. %-44s %.5f (validado: %.5f)  ->  %s" % (k, r, v, "OK" if round(r, 5) == v else "REVISAR"))

## 9 · Tabla comparativa

**Orden:** RMSE (raíz del error cuadrático medio) promedio en los repartos 42 a 51, de menor a mayor. Errores en t/ha (toneladas por hectárea).

- **Desv.:** cuánto cambia el RMSE de un reparto a otro (estabilidad).
- **Gana vs. municipal:** en cuántos de los 10 repartos tiene menor RMSE que el promedio municipal.
- **Remuestreo:** en qué porcentaje de 2,000 grupos de parcelas sacados al azar mejora al promedio municipal.

In [ ]:
def met(s, m):
    p = PRED[s][m]
    return np.sqrt(mean_squared_error(y_ent, p)), mean_absolute_error(y_ent, p), r2_score(y_ent, p)
IDX = np.random.default_rng(SEMILLA).integers(0, len(y_ent), (N_BOOT, len(y_ent)))
def remuestreo(m, ref="Promedio municipal"):
    em = np.stack([(y_ent - PRED[s][m]) ** 2 for s in SEMILLAS]); er = np.stack([(y_ent - PRED[s][ref]) ** 2 for s in SEMILLAS])
    dif = (np.sqrt(er[:, IDX].mean(axis=2)) - np.sqrt(em[:, IDX].mean(axis=2))).mean(axis=0)
    return np.percentile(dif, 2.5), np.percentile(dif, 97.5), 100 * (dif > 0).mean()

RM = {m: np.array([met(s, m) for s in SEMILLAS]) for m in NOMBRES}
RN = {m: np.array([met(s, m) for s in SEMILLAS_NUEVAS]) for m in NOMBRES}
filas = []
for m in NOMBRES:
    M, Mn, Ref = RM[m], RN[m], RM["Promedio municipal"]
    f = {"modelo": m, "RMSE": M[:, 0].mean(), "MAE": M[:, 1].mean(), "R²": M[:, 2].mean(), "desv. RMSE": M[:, 0].std(ddof=1),
         "gana vs. municipal": "%d de 10" % (M[:, 0] < Ref[:, 0]).sum(), "RMSE nuevos": Mn[:, 0].mean(),
         "gana vs. municipal (nuevos)": "%d de 10" % (Mn[:, 0] < RN["Promedio municipal"][:, 0]).sum()}
    if m != "Promedio municipal":
        lo, hi, pc = remuestreo(m)
        f["remuestreo: mejora en"], f["intervalo 95 %"] = "%.1f %%" % pc, "%.4f a %.4f" % (lo, hi)
    filas.append(f)
TABLA_CRUDA = pd.DataFrame(filas).sort_values("RMSE").reset_index(drop=True); TABLA_CRUDA.index += 1
TABLA = TABLA_CRUDA.copy()
num = ["RMSE", "MAE", "R²", "desv. RMSE", "RMSE nuevos"]; TABLA[num] = TABLA[num].round(5)
print(TABLA.fillna("—").to_string())
TABLA.to_csv("comparacion_modelos_final.csv", index_label="lugar", encoding="utf-8-sig")
if COMPARAR:
    pesos = {}
    for s in SEMILLAS:
        for e in ELEG[s]:
            for k, v in e.items():
                if v.startswith("w="): pesos.setdefault(k, []).append(float(v[2:]))
    print("\nPeso w del primer modelo de cada pareja (promedio en los 50 pliegues):",
          {k: round(float(np.mean(v)), 2) for k, v in pesos.items()})
    print("Penalización elegida por la LSTM:", pd.Series([e["LSTM"][2] if isinstance(e["LSTM"], tuple) else e["LSTM"] for s in SEMILLAS for e in ELEG[s]]).value_counts().to_dict())

## 10 · Selección automática del ganador

Se aplica la regla, paso por paso, a **todos** los modelos por igual:
1. **Menor RMSE.**
2. **Empate:** un modelo empata con el mejor si la diferencia promedio de RMSE frente a él es menor que lo que esa diferencia varía de un reparto a otro. Entre los empatados, gana el de **menor MAE**.
3. Si en MAE también empatan (mismo criterio), gana el de **menor variación** del RMSE.

In [ ]:
def empatan(candidatos, mejor, col):
    """Modelos cuya diferencia promedio con 'mejor' es menor que la variación de esa diferencia entre repartos."""
    k = {"RMSE": 0, "MAE": 1}[col]
    salida = [mejor]
    for m in candidatos:
        if m == mejor: continue
        dif = RM[m][:, k] - RM[mejor][:, k]
        if dif.mean() < dif.std(ddof=1):
            salida.append(m)
    return salida

orden = TABLA_CRUDA.sort_values("RMSE").modelo.tolist()
paso1 = orden[0]
E1 = empatan(orden, paso1, "RMSE")
print("Paso 1 · menor RMSE:", paso1, "(%.5f)" % RM[paso1][:, 0].mean())
print("        empatan en RMSE con él:", E1)
if len(E1) == 1:
    GANADOR = paso1
else:
    paso2 = min(E1, key=lambda m: RM[m][:, 1].mean())
    E2 = [m for m in empatan(E1, paso2, "MAE") if m in E1]
    print("Paso 2 · menor MAE entre los empatados:", paso2, "(%.5f)" % RM[paso2][:, 1].mean())
    print("        empatan en MAE con él:", E2)
    GANADOR = paso2 if len(E2) == 1 else min(E2, key=lambda m: RM[m][:, 0].std(ddof=1))
    if len(E2) > 1:
        print("Paso 3 · menor variación entre los empatados:", GANADOR)
print("\nGANADOR SELECCIONADO POR LA REGLA:", GANADOR)
print("¿Coincide con el ganador validado antes?", "sí" if GANADOR == GANADOR_VALIDADO else "NO: revisar antes de seguir")

# Tabla de los 4 finalistas (por RMSE), con lugar en cada métrica; los empates se marcan
FIN = orden[:4]
col = {1: "🟢", 2: "🟡", 3: "🟠", 4: "🔴"}
def lugares(k, menor_mejor=True):
    v = {m: RM[m][:, k].mean() for m in FIN}
    return {m: sorted(v.values(), reverse=not menor_mejor).index(v[m]) + 1 for m in FIN}
lr, lm, l2 = lugares(0), lugares(1), lugares(2, menor_mejor=False)
ld = {m: sorted([RM[x][:, 0].std(ddof=1) for x in FIN]).index(RM[m][:, 0].std(ddof=1)) + 1 for m in FIN}
FINALISTAS = pd.DataFrame([{"modelo": m, "RMSE": "%s %.5f" % (col[lr[m]], RM[m][:, 0].mean()), "MAE": "%s %.5f" % (col[lm[m]], RM[m][:, 1].mean()),
                            "R²": "%s %.5f" % (col[l2[m]], RM[m][:, 2].mean()), "desv. RMSE": "%s %.5f" % (col[ld[m]], RM[m][:, 0].std(ddof=1)),
                            "empata en RMSE con el 1.º": "sí" if (m in E1 and m != paso1) else ("—" if m == paso1 else "no")} for m in FIN])
print("\nFinalistas (colores: lugar en cada métrica; revisar la columna de empate antes de leer los colores):")
print(FINALISTAS.to_string(index=False))

## 11 · Comprobación del ganador

Compara las cifras del ganador con las ya validadas. Si todo dice **OK**, es exactamente el mismo modelo.

In [ ]:
ESPERADO = {GANADOR_VALIDADO: {"comp": (0.49479, 0.33194, 0.65956), "nuevos": (0.49330, 0.32849, 0.66160)}}
if GANADOR in ESPERADO:
    ok = True
    for etq, R in (("comp", RM[GANADOR]), ("nuevos", RN[GANADOR])):
        r = tuple(round(x, 5) for x in R.mean(axis=0)); e = ESPERADO[GANADOR][etq]
        print("%-7s RMSE %.5f · MAE %.5f · R² %.5f  ->  %s" % (etq, *r, "OK" if r == e else "REVISAR")); ok &= r == e
    lo, hi, pc = remuestreo(GANADOR)
    print("Remuestreo: mejora al municipal en %.1f %% (validado: 99.1 %%)  ->  %s" % (pc, "OK" if round(pc, 1) == 99.1 else "REVISAR"))
    print("\nCOMPROBACIÓN:", "TODO COINCIDE" if ok and round(pc, 1) == 99.1 else "HAY DIFERENCIAS: avisa antes de seguir")
else:
    print("El ganador no es el validado antes: no hay cifras previas para comparar. Revisar con el equipo.")
err = np.abs(np.array([y_ent - PRED[s][GANADOR] for s in SEMILLAS]))
PCT05, PCT10, R2 = 100 * (err <= 0.5).mean(), 100 * (err <= 0.1 * y_ent).mean(), 100 * RM[GANADOR][:, 2].mean()
print("Error ≤ 0.5 t/ha: %.1f %% · error ≤ 10 %% del rendimiento: %.1f %% · R²: %.1f %%" % (PCT05, PCT10, R2))
print("Estimación propia de confiabilidad (promedio de esos tres; NO es métrica oficial): %.1f %%" % np.mean([PCT05, PCT10, R2]))

## 12 · Entrenamiento final y las 59 predicciones

👉 **Aquí se entrena por última vez, con las 138 parcelas completas,** usando el procedimiento del ganador, y se predicen las 59. En Nanacamilpa, el promedio municipal es el rendimiento de su única parcela de entrenamiento.

**Archivo de entrega:** `visionterradata_prediccion.csv`, con la estructura del archivo de FIRA (Fideicomisos Instituidos en Relación con la Agricultura):
1. Columnas `ID_POLIGONO`, `AREA_HA`, `RENDIMIENTO_T_HA` y `CONJUNTO`, en ese orden.
2. Solo las 59 parcelas de PREDICCION, de menor a mayor identificador.
3. Identificador, área y conjunto copiados tal cual del archivo de FIRA; el rendimiento es la predicción del modelo, con cuatro decimales.
4. UTF-8 (formato de transformación Unicode de 8 bits) con BOM (marca de orden de bytes) y renglones terminados en CRLF (retorno de carro y salto de línea), como el archivo de FIRA, pero sin la coma al final de cada renglón.

Antes de guardarlo, el cuaderno lo revisa completo; si algo falla, se detiene y no lo guarda. También imprime la Tabla 3 del anexo y la media predicha por municipio de la Tabla 2, listas para copiar.

In [ ]:
pr_idx = IDX_PRED
pred_final, _ = ajustar_todo(IDX_ENT, IDX_PRED)          # entrena solo con las 138 y predice las 59
p59, g59 = pred_final[GANADOR], G_ALL[IDX_PRED]
fallos = []
if len(p59) != 59: fallos.append("no son 59")
if np.isnan(p59).any(): fallos.append("hay vacíos")
if fallos: raise Exception("REVISIÓN FALLIDA: " + ", ".join(fallos))
print("Ganador:", GANADOR)
print("Revisión: OK · 59 predicciones · %.4f a %.4f t/ha (entrenamiento: %.2f a %.2f) · fuera de rango: %d"
      % (p59.min(), p59.max(), y.min(), y.max(), int(((p59 < y.min()) | (p59 > y.max())).sum())))

# --- CSV de entrega con la estructura del archivo de FIRA: solo las 59 de PREDICCION, de menor a mayor identificador ---
FORMATO = "%." + str(DECIMALES) + "f"                       # siempre con DECIMALES decimales: 3.1000, no 3.1
ids59 = d.ID_POLIGON.values[pr_idx]
P59 = dict(zip(ids59, p59))                                 # predicción sin redondear de cada parcela
R59 = {i: np.round(v, DECIMALES) for i, v in P59.items()}   # la misma, redondeada a DECIMALES
F59 = fira[fira.CONJUNTO == "PREDICCION"].sort_values("ID_POLIGONO").reset_index(drop=True)   # identificador, área y conjunto tal cual
F59["RENDIMIENTO_T_HA"] = [FORMATO % R59[i] for i in F59.ID_POLIGONO]
FIN, BOM = "\r\n", b"\xef\xbb\xbf"                           # CRLF (retorno de carro y salto de línea) y BOM (marca de orden de bytes), como FIRA
TEXTO = FIN.join([",".join(COLUMNAS_FIRA)] + [",".join(r) for r in F59[COLUMNAS_FIRA].values.tolist()]) + FIN
BYTES_CSV = BOM + TEXTO.encode("utf-8")

# Revisión completa antes de guardar: si algo falla, se detiene y no guarda
L_ = pd.read_csv(io.BytesIO(BYTES_CSV), dtype=str, keep_default_na=False, encoding="utf-8-sig")   # se lee como lo leería FIRA
O_ = fira[fira.CONJUNTO == "PREDICCION"].set_index("ID_POLIGONO")
huella_2d = hashlib.md5("\n".join(sorted("%s:%.2f" % (i, v) for i, v in zip(ids59, np.round(p59, 2)))).encode("utf-8")).hexdigest()
renglones = TEXTO.split(FIN)[:-1]
revision = {
    "encabezado: " + ",".join(COLUMNAS_FIRA): list(L_.columns) == COLUMNAS_FIRA,
    "59 renglones, uno por parcela, sin repetidos": len(L_) == 59 and not L_.ID_POLIGONO.duplicated().any(),
    "las mismas 59 parcelas de PREDICCION del archivo de FIRA": set(L_.ID_POLIGONO) == set(O_.index) == set(ids59),
    "de menor a mayor identificador": list(L_.ID_POLIGONO) == sorted(L_.ID_POLIGONO),
    "área y conjunto copiados tal cual del archivo de FIRA": list(L_.AREA_HA) == list(O_.loc[L_.ID_POLIGONO, "AREA_HA"]) and set(L_.CONJUNTO) == {"PREDICCION"},
    "rendimiento con %d decimales, sin vacíos" % DECIMALES: bool(L_.RENDIMIENTO_T_HA.str.fullmatch(r"\d+\.\d{%d}" % DECIMALES).all()),
    "cada rendimiento es la predicción del modelo para su parcela": bool(np.abs(pd.to_numeric(L_.RENDIMIENTO_T_HA, errors="coerce").values
        - np.array([P59[i] for i in L_.ID_POLIGONO])).max() <= 0.5 * 10 ** -DECIMALES + 1e-12),
    "mismas predicciones que las validadas, a dos decimales (huella md5)": huella_2d == HUELLA_VALIDADA_2D,
    "UTF-8 con BOM, renglones con CRLF y sin coma al final": BYTES_CSV.startswith(BOM) and TEXTO.count("\n") == TEXTO.count("\r") == TEXTO.count(FIN) == 60
        and all(r.count(",") == 3 and not r.endswith(",") for r in renglones),
}
malas = [k for k, v in revision.items() if not v]
if malas:
    raise Exception("REVISIÓN DEL CSV FALLIDA; el archivo NO se guardó:\n- " + "\n- ".join(malas)
                    + "\nGanador: %s (validado: %s). Revisa que COMPARAR = True y que los archivos sean los originales." % (GANADOR, GANADOR_VALIDADO))
ARCHIVO_FIRA = "%s_prediccion.csv" % NOMBRE_EQUIPO
with open(ARCHIVO_FIRA, "wb") as fh:                         # modo binario: se guardan exactamente los bytes revisados
    fh.write(BYTES_CSV)
with open(ARCHIVO_FIRA, "rb") as fh:
    if fh.read() != BYTES_CSV: raise Exception("El archivo guardado no coincide con el revisado")
print("\nRevisión del CSV antes de guardarlo:")
for k in revision: print("  OK ·", k)
print("Guardado: %s · %d bytes · huella md5 %s" % (ARCHIVO_FIRA, len(BYTES_CSV), hashlib.md5(BYTES_CSV).hexdigest()))
SALIDA = pd.DataFrame({COLUMNA_ID: F59.ID_POLIGONO, COLUMNA_REND: F59.RENDIMIENTO_T_HA})   # mismos valores y orden que el CSV de entrega
SALIDA.to_csv("tabla_anexos_predicciones.csv", index=False, encoding="utf-8")
print("Guardados: tabla_anexos_predicciones.csv ·", ARCHIVO_FIRA, "· comparacion_modelos_final.csv")

# --- Tabla 3 del anexo: dos pares de columnas (30 + 29 renglones), separadas por tabuladores para pegar en Word ---
n1 = (len(F59) + 1) // 2
print("\nTabla 3 · Rendimiento predicho de las 59 parcelas, en t/ha (toneladas por hectárea), con %d decimales:" % DECIMALES)
print("\t".join([COLUMNA_ID, "Rendimiento (t/ha)"] * 2))
for k in range(n1):
    der = [F59.ID_POLIGONO[k + n1], F59.RENDIMIENTO_T_HA[k + n1]] if k + n1 < len(F59) else ["", ""]
    print("\t".join([F59.ID_POLIGONO[k], F59.RENDIMIENTO_T_HA[k]] + der))

# --- Tabla 2 del anexo: media predicha por municipio, con los mismos valores de la Tabla 3 ---
T2 = pd.DataFrame({"Estado": d.Estado.values[pr_idx], "Municipio": g59, "pred": [R59[i] for i in ids59]})
T2 = T2.groupby(["Estado", "Municipio"])["pred"].agg(["size", "mean"]).reset_index()
print("\nTabla 2 · Media predicha por municipio, en t/ha, promedio de los valores de la Tabla 3:")
print("\t".join(["Estado", "Municipio", "n pred.", "Media pred. (t/ha)"]))
for e_, m_, n_, v_ in T2.values.tolist():
    print("\t".join([e_, m_, str(n_), FORMATO % v_]))

# Qué variables usa el ganador y cuánto pesa cada una (ajuste final con las 138 parcelas)
MESES = {4: "abr", 5: "may", 6: "jun", 7: "jul", 8: "ago", 9: "sep", 10: "oct"}
def etiqueta(c):
    if c in ("elev_mean", "pend_mean"): return {"elev_mean": "elevación", "pend_mean": "pendiente"}[c]
    v, _, m, _ = c.split("_"); return "%s %s" % ({"prec": "lluvia", "tmin": "tmín", "tmax": "tmáx"}[v], MESES.get(int(m), m))
GRUPO_DE = {"Promedio municipal + ½ temperatura máxima mensual": COL_TMAX, "Promedio municipal + ½ temperatura mínima mensual": COL_TMIN,
            "Promedio municipal + ½ temperatura mensual (máx. y mín.)": COL_TEMP, CLI: COL_CLIMA,
            "Promedio municipal + ½ topografía": TOPO, "Promedio municipal + ½ lluvia mensual": COL_LLUVIA}
if GANADOR in GRUPO_DE:
    cols = GRUPO_DE[GANADOR]; Xg = d[cols].values.astype(float)[IDX_ENT]
    mejor = None
    for l1 in MEZCLAS:                                    # misma elección interna que 'lineal'
        alfas = _rejilla(Xg, y_ent, g_ent, l1); oof = np.zeros((len(y_ent), len(alfas)))
        for a_, b_ in internos(len(y_ent)):
            oof[b_] = _pred_lin(_camino(Xg[a_], y_ent[a_], g_ent[a_], alfas, l1), Xg[b_], g_ent[b_])
        e = ((oof - y_ent[:, None]) ** 2).sum(axis=0); k = int(e.argmin())
        if mejor is None or e[k] < mejor[0]: mejor = (e[k], l1, alfas[k])
    B_ = _camino(Xg, y_ent, g_ent, np.array([mejor[2]]), mejor[1])["B"][:, 0]
    sd = pd.DataFrame(Xg).groupby(g_ent).transform(lambda c: c - c.mean()).std().values
    COEF = pd.DataFrame({"variable": [etiqueta(c) for c in cols], "efecto por desviación en la predicción (t/ha)": 0.5 * B_ * sd + 0.0})   # + 0.0: un cero sale 0.0000, no -0.0000
    COEF = COEF.reindex(COEF.iloc[:, 1].abs().sort_values(ascending=False).index)
    print("\nVariables del ganador · efecto de subir una desviación dentro de su municipio (ya incluye el ½):")
    print(COEF.round(4).to_string(index=False))
    print("Variables con efecto distinto de cero: %d de %d" % ((COEF.iloc[:, 1].abs() > 1e-9).sum(), len(cols)))

## 13 · Gráficas para el reporte

👉 **Aquí se generan las visualizaciones del ganador.**
1. **Mapa de calor:** el rendimiento y las variables del ganador, con todas las parcelas juntas y dentro de cada municipio.
2. **Reales contra predichos:** parcelas no vistas, reparto 42.
3. **Errores:** dónde (por municipio) y cuánto se equivoca, también en el reparto 42.
4. **Las 59 predicciones** frente al rendimiento de entrenamiento de su municipio.

In [ ]:
GRAF = []
cols = GRUPO_DE.get(GANADOR, TOPO)
T3 = pd.DataFrame(d[cols].values.astype(float)[IDX_ENT], columns=[etiqueta(c) for c in cols]); T3.insert(0, "rendimiento", y_ent)
dentro = T3.groupby(g_ent).transform(lambda c: c - c.mean())
n = T3.shape[1]; letra = 11 if n <= 4 else 6
fig, axs = plt.subplots(1, 2, figsize=(14 if n > 4 else 11, 6.4 if n > 4 else 4.6))
for ax, (tit, M) in zip(axs, (("Todas las parcelas juntas", T3.corr()), ("Dentro de cada municipio", dentro.corr()))):
    im = ax.imshow(M.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(M.columns, rotation=90 if n > 4 else 0, fontsize=8)
    ax.set_yticks(range(n)); ax.set_yticklabels(M.columns, fontsize=8)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, "%.2f" % M.values[i, j], ha="center", va="center", fontsize=letra,
                    color="white" if abs(M.values[i, j]) > 0.7 else "black")
    ax.set_title(tit)
fig.colorbar(im, ax=axs, fraction=0.03, label="correlación"); fig.suptitle("Correlaciones de las variables del modelo ganador (138 parcelas)")
fig.savefig("g1_mapa_calor.png", dpi=150, bbox_inches="tight"); plt.show(); GRAF.append("g1_mapa_calor.png")

p42 = PRED[SEMILLA][GANADOR]
lim = [1.8, 5.8]
fig, ax = plt.subplots(figsize=(6.4, 6.2))
for m in pd.Series(g_ent).value_counts().index:
    k = g_ent == m; ax.scatter(y_ent[k], p42[k], s=28, alpha=0.8, label="%s (%d)" % (m, k.sum()))
ax.plot(lim, lim, "k-", lw=1, label="predicción perfecta")
ax.fill_between(lim, [lim[0] - 0.5, lim[1] - 0.5], [lim[0] + 0.5, lim[1] + 0.5], color="grey", alpha=0.15, label="± 0.5 t/ha")
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel("Rendimiento real (t/ha)"); ax.set_ylabel("Rendimiento predicho (t/ha)")
ax.set_title("Reales contra predichos · parcelas no vistas (reparto 42)")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))       # fuera de la gráfica: no tapa puntos
fig.savefig("g2_reales_vs_predichos.png", dpi=150, bbox_inches="tight"); plt.show(); GRAF.append("g2_reales_vs_predichos.png")

res = y_ent - p42
ordm = pd.Series(g_ent).value_counts().index
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
axs[0].boxplot([res[g_ent == m] for m in ordm])
axs[0].set_xticks(range(1, len(ordm) + 1)); axs[0].set_xticklabels(["%s (%d)" % (m, (g_ent == m).sum()) for m in ordm], fontsize=8, rotation=35, ha="right")
axs[0].axhline(0, color="k", lw=1); axs[0].set_ylabel("Error = real − predicho (t/ha)"); axs[0].set_title("Dónde se equivoca: error por municipio")
axs[1].hist(np.abs(res), bins=20, color="#4C8C6B", edgecolor="white"); axs[1].axvline(0.5, color="k", ls="--", label="0.5 t/ha")
axs[1].set_xlabel("Error absoluto (t/ha)"); axs[1].set_ylabel("Número de parcelas"); axs[1].set_title("Cuánto se equivoca"); axs[1].legend()
fig.suptitle("Parcelas no vistas · reparto 42")
fig.savefig("g3_errores.png", dpi=150, bbox_inches="tight"); plt.show(); GRAF.append("g3_errores.png")

mun59 = [m for m in ordm if (g59 == m).any()]
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.boxplot([y_ent[g_ent == m] for m in mun59])
for i, m in enumerate(mun59, start=1):
    k = g59 == m
    ax.scatter(np.full(k.sum(), i) + np.random.default_rng(0).uniform(-0.15, 0.15, k.sum()), p59[k], color="#C0392B", s=18, zorder=3)
ax.set_xticks(range(1, len(mun59) + 1)); ax.set_xticklabels(["%s\n(%d a predecir)" % (textwrap.fill(m, 16), (g59 == m).sum()) for m in mun59], fontsize=8)
ax.set_ylabel("t/ha"); ax.set_title("Cajas: rendimiento real de entrenamiento · puntos rojos: las 59 predicciones")
fig.savefig("g4_predicciones_59.png", dpi=150, bbox_inches="tight"); plt.show(); GRAF.append("g4_predicciones_59.png")

try:
    from google.colab import files
    for f in GRAF + ["comparacion_modelos_final.csv", "tabla_anexos_predicciones.csv", ARCHIVO_FIRA, "mapa_calor_22.png"]:
        files.download(f)
except ImportError:
    pass
print("Gráficas guardadas:", GRAF)

## 14 · Comparar contra los rendimientos reales (cuando FIRA los entregue)

1. Esta celda crea `plantilla_rendimientos_reales.csv`, con las 59 parcelas y una columna vacía.
2. Cuando haya rendimientos reales, llénala o usa el archivo de FIRA (identificador y rendimiento).
3. Súbelo y ejecuta `comparar_con_reales("nombre_del_archivo.csv")`.

In [ ]:
pd.DataFrame({"ID_POLIGON": d.ID_POLIGON.values[pr_idx], "rendimiento_real_t_ha": np.nan}).to_csv("plantilla_rendimientos_reales.csv", index=False)
print("Plantilla creada: plantilla_rendimientos_reales.csv")

def comparar_con_reales(archivo, col_id=None, col_real=None):
    r = pd.read_csv(archivo)
    col_id = col_id or next(c for c in r.columns if "id" in c.lower())
    col_real = col_real or next(c for c in r.columns if c != col_id)
    t = pd.DataFrame({"ID_POLIGON": d.ID_POLIGON.values[pr_idx], "predicho": p59}).merge(
        r[[col_id, col_real]].rename(columns={col_id: "ID_POLIGON", col_real: "real"}), on="ID_POLIGON").dropna()
    if len(t) == 0:
        raise Exception("No hay rendimientos reales para comparar")
    rm, ma, r2 = np.sqrt(mean_squared_error(t.real, t.predicho)), mean_absolute_error(t.real, t.predicho), r2_score(t.real, t.predicho)
    print("Parcelas comparadas: %d · RMSE %.4f · MAE %.4f · R² %.4f" % (len(t), rm, ma, r2))
    print("Esperado según la validación: RMSE cerca de %.2f t/ha" % RM[GANADOR][:, 0].mean())
    fig, ax = plt.subplots(figsize=(5.5, 5.3)); ax.scatter(t.real, t.predicho, s=25)
    ax.plot(lim, lim, "k-", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Real (t/ha)"); ax.set_ylabel("Predicho (t/ha)"); ax.set_title("59 parcelas: reales contra predichos"); plt.show()
    return t
# resultado = comparar_con_reales("plantilla_rendimientos_reales.csv")

## 15 · Información incorporada

In [ ]:
USO = pd.DataFrame([
    ["todo__basico_terminado.csv", "Rendimiento, municipio, topografía, clima e índices mensuales de 2025",
     "Rehacer las 22 variables, armar las secuencias de la LSTM y entrenar", "Ninguna al archivo; las variables se calculan con las funciones de las Fases 1 y 2"],
    ["ID_area_rendimiento_70_30_Reto_AgroCebada.csv", "Archivo original de FIRA: identificador, área, rendimiento y conjunto de las 197 parcelas",
     "Comprobar que coincide con la tabla maestra y copiar tal cual el identificador, el área y el conjunto de las 59 parcelas de PREDICCION en el CSV de entrega",
     "Ninguna al archivo; el CSV de entrega se escribe aparte, con las mismas columnas y sin la coma final de cada renglón"],
    ["candidatas_fase2.csv (opcional)", "Las 22 variables de la Fase 2",
     "Verificar que las 22 rehechas sean idénticas", "Ninguna"],
    ["diccionario_1381_completo.csv (opcional)", "Significado de las columnas mensuales",
     "Describir los 7 datos por mes de la LSTM", "Ninguna"]],
    columns=["archivo", "qué se usó", "para qué", "modificaciones"])
USO